In [1]:
# ==============================================================================
# PROJECT TITLE: BioShield Web - Automated HTTP Header & CORS Security Auditor
# AUTHOR: Sampriti Roy
# COURSE: IBM SkillsBuild Cyber Security BootCamp 2026 | BharatCares
# DESCRIPTION: A 100% cross-platform Python security scanner that audits websites
#              for missing HTTP Security Headers and CORS (Cross-Origin Resource
#              Sharing) misconfigurations, generating a modern dark-themed report.
# ==============================================================================

import requests
import urllib3
from datetime import datetime

# Disable SSL warnings so beginner tests on local/test sites don't show clutter
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Check if running inside Google Colab / Jupyter Notebook for rich dark UI display
try:
    from IPython.display import display, HTML
    IN_NOTEBOOK = True
except ImportError:
    IN_NOTEBOOK = False


# ------------------------------------------------------------------------------
# STEP 1: DEFINE THE "IMMUNE SYSTEM" CHECKLIST (SECURITY HEADERS)
# Biology Analogy: Think of a website like a living cell. HTTP Security Headers
# act like the selectively permeable cell membrane—blocking toxic molecules
# (malicious scripts) while allowing healthy nutrients (safe user traffic) inside.
# ------------------------------------------------------------------------------
SECURITY_HEADERS = {
    "Strict-Transport-Security": {
        "purpose": "Enforces encrypted HTTPS connections (HSTS).",
        "bio_analogy": "Outer Cell Wall: Prevents eavesdropping in transit.",
        "risk": "High"
    },
    "Content-Security-Policy": {
        "purpose": "Blocks unauthorized scripts & Cross-Site Scripting (XSS).",
        "bio_analogy": "Immune Receptor: Rejects foreign viral DNA/scripts.",
        "risk": "High"
    },
    "X-Frame-Options": {
        "purpose": "Prevents Clickjacking (hiding the site inside invisible frames).",
        "bio_analogy": "Structural Cytoskeleton: Stops cell-mimicry traps.",
        "risk": "Medium"
    },
    "X-Content-Type-Options": {
        "purpose": "Stops browsers from guessing/sniffing dangerous file types.",
        "bio_analogy": "Enzyme Specificity: Ensures only exact file types execute.",
        "risk": "Medium"
    },
    "Referrer-Policy": {
        "purpose": "Controls how much user navigation history is leaked to other sites.",
        "bio_analogy": "Membrane Transport Gate: Limits internal data leakage.",
        "risk": "Low"
    },
    "Permissions-Policy": {
        "purpose": "Restricts unauthorized access to camera, microphone, and GPS.",
        "bio_analogy": "Organelle Lock: Protects sensitive hardware functions.",
        "risk": "Low"
    }
}


# ------------------------------------------------------------------------------
# STEP 2: THE SCANNER FUNCTION (AUDITS HEADERS & CORS CONFIGURATION)
# ------------------------------------------------------------------------------
def audit_website(url):
    # Ensure URL starts with https://
    if not url.startswith("http://") and not url.startswith("https://"):
        url = "https://" + url

    # We send a test 'Origin' header to safely inspect how the server's CORS policy responds
    test_origin = "https://unauthorized-external-site.example"
    request_headers = {
        "User-Agent": "BioShield-Security-Auditor/1.0 (Educational Project)",
        "Origin": test_origin
    }

    try:
        response = requests.get(url, headers=request_headers, timeout=10, verify=True)
        resp_headers = response.headers

        header_results = []
        score = 100

        # 1. Check all 6 Security Headers
        for header_name, info in SECURITY_HEADERS.items():
            if header_name in resp_headers:
                header_results.append({
                    "header": header_name,
                    "status": "PRESENT (SAFE)",
                    "badge": "pass",
                    "value": resp_headers[header_name][:55] + ("..." if len(resp_headers[header_name]) > 55 else ""),
                    "details": info["purpose"] + f" ({info['bio_analogy']})"
                })
            else:
                # Deduct points based on risk level if missing
                penalty = 15 if info["risk"] == "High" else (10 if info["risk"] == "Medium" else 5)
                score -= penalty
                header_results.append({
                    "header": header_name,
                    "status": f"MISSING ({info['risk']} Risk)",
                    "badge": "fail" if info["risk"] == "High" else "warn",
                    "value": "Not Configured",
                    "details": info["purpose"] + f" ({info['bio_analogy']})"
                })

        # 2. Passive Audit of Cross-Origin Resource Sharing (CORS) Policy
        acao = resp_headers.get("Access-Control-Allow-Origin", "Not Present")
        acac = resp_headers.get("Access-Control-Allow-Credentials", "false").lower()

        if acao == test_origin and acac == "true":
            cors_status = "CRITICAL FLAW: Reflects arbitrary Origin with Credentials"
            cors_badge = "fail"
            score -= 25
        elif acao == test_origin:
            cors_status = "WARNING: Reflects arbitrary Origin header"
            cors_badge = "warn"
            score -= 15
        elif acao == "*":
            cors_status = "PUBLIC WILDCARD (*): Open to all domains (Verify no private data is exposed)"
            cors_badge = "warn"
            score -= 5
        else:
            cors_status = "SECURE: Restrictive or No Wildcard CORS Policy"
            cors_badge = "pass"

        return {
            "url": url,
            "status_code": response.status_code,
            "score": max(0, score),
            "headers": header_results,
            "cors_header": acao,
            "cors_status": cors_status,
            "cors_badge": cors_badge,
            "error": None
        }

    except Exception as e:
        return {"url": url, "error": str(e)}


# ------------------------------------------------------------------------------
# STEP 3: MODERN DARK-THEME DASHBOARD GENERATOR (HTML/CSS)
# ------------------------------------------------------------------------------
def generate_dark_dashboard(scan_data_list):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    html = f"""
    <style>
        .bioshield-app {{
            background-color: #0d1117;
            color: #c9d1d9;
            font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
            padding: 24px;
            border-radius: 14px;
            border: 1px solid #30363d;
            max-width: 950px;
            margin: 10px auto;
            box-shadow: 0 10px 30px rgba(0,0,0,0.6);
        }}
        .header-banner {{
            border-bottom: 1px solid #21262d;
            padding-bottom: 16px;
            margin-bottom: 20px;
        }}
        .header-banner h1 {{
            color: #58a6ff;
            margin: 0 0 6px 0;
            font-size: 24px;
        }}
        .header-banner p {{
            color: #8b949e;
            margin: 0;
            font-size: 13.5px;
        }}
        .site-card {{
            background-color: #161b22;
            border: 1px solid #30363d;
            border-radius: 10px;
            padding: 18px;
            margin-bottom: 22px;
        }}
        .site-header {{
            display: flex;
            justify-content: space-between;
            align-items: center;
            flex-wrap: wrap;
            border-bottom: 1px solid #21262d;
            padding-bottom: 12px;
            margin-bottom: 14px;
        }}
        .site-title {{
            font-size: 17px;
            font-weight: bold;
            color: #f0f6fc;
        }}
        .score-pill {{
            padding: 6px 14px;
            border-radius: 20px;
            font-weight: bold;
            font-size: 13px;
        }}
        .pass {{ background-color: rgba(35, 134, 54, 0.2); color: #3fb950; border: 1px solid #238636; }}
        .warn {{ background-color: rgba(187, 128, 9, 0.2); color: #d29922; border: 1px solid #9e6a03; }}
        .fail {{ background-color: rgba(218, 54, 51, 0.2); color: #f85149; border: 1px solid #da3633; }}

        table.audit-table {{
            width: 100%;
            border-collapse: collapse;
            font-size: 13px;
            margin-top: 10px;
        }}
        table.audit-table th {{
            text-align: left;
            padding: 10px;
            background-color: #0d1117;
            color: #8b949e;
            border-bottom: 1px solid #30363d;
        }}
        table.audit-table td {{
            padding: 10px;
            border-bottom: 1px solid #21262d;
            vertical-align: top;
        }}
        .cors-box {{
            margin-top: 14px;
            padding: 12px;
            border-radius: 8px;
            background-color: #0d1117;
            border: 1px solid #30363d;
            font-size: 13px;
        }}
        .badge-sm {{
            padding: 3px 8px;
            border-radius: 6px;
            font-size: 11px;
            font-weight: 600;
            display: inline-block;
        }}
    </style>

    <div class="bioshield-app">
        <div class="header-banner">
            <h1>🛡️ BioShield Web: Automated Header & CORS Auditor</h1>
            <p>Cross-Platform Security Assessment Tool | Audited on: {timestamp} | Author: Sampriti Roy</p>
        </div>
    """

    for data in scan_data_list:
        if data.get("error"):
            html += f"""
            <div class="site-card">
                <div class="site-title">🌐 {data['url']}</div>
                <p style="color:#f85149;">❌ Connection Error: {data['error']}</p>
            </div>
            """
            continue

        score = data["score"]
        score_class = "pass" if score >= 80 else ("warn" if score >= 50 else "fail")

        html += f"""
        <div class="site-card">
            <div class="site-header">
                <div class="site-title">🌐 Target: {data['url']} <span style="color:#8b949e;font-size:13px;">(HTTP {data['status_code']})</span></div>
                <div class="score-pill {score_class}">Security Score: {score} / 100</div>
            </div>

            <table class="audit-table">
                <thead>
                    <tr>
                        <th style="width:24%;">Security Header</th>
                        <th style="width:20%;">Audit Status</th>
                        <th style="width:24%;">Configured Value</th>
                        <th style="width:32%;">Role & Bio-Defense Analogy</th>
                    </tr>
                </thead>
                <tbody>
        """

        for h in data["headers"]:
            html += f"""
                    <tr>
                        <td style="font-weight:600;color:#79c0ff;">{h['header']}</td>
                        <td><span class="badge-sm {h['badge']}">{h['status']}</span></td>
                        <td style="font-family:monospace;color:#8b949e;">{h['value']}</td>
                        <td style="color:#b1bac4;">{h['details']}</td>
                    </tr>
            """

        html += f"""
                </tbody>
            </table>

            <div class="cors-box">
                <strong>🔄 CORS (Cross-Origin Resource Sharing) Audit:</strong><br>
                <span style="color:#8b949e;">Access-Control-Allow-Origin Value:</span> <code>{data['cors_header']}</code> &nbsp;|&nbsp;
                <span class="badge-sm {data['cors_badge']}">{data['cors_status']}</span>
            </div>
        </div>
        """

    html += "</div>"
    return html


# ------------------------------------------------------------------------------
# STEP 4: RUN THE SCANNER (DEFAULT BENCHMARK TARGETS)
# We test 3 safe, well-known reference websites including NCBI (Biotech database)
# ------------------------------------------------------------------------------
if __name__ == "__main__":
    # List of safe public sites to audit (includes NCBI for Biotechnology relevance!)
    target_websites = [
        "https://www.ncbi.nlm.nih.gov",  # National Center for Biotechnology Information
        "https://github.com",            # Industry Gold-Standard Security Headers
        "http://example.com"             # Standard IANA Test Domain (Missing several headers)
    ]

    print("🔍 Starting BioShield Web Security Scanner...")
    all_results = []

    for site in target_websites:
        print(f" -> Auditing: {site} ...")
        result = audit_website(site)
        all_results.append(result)

    dashboard_html = generate_dark_dashboard(all_results)

    # If in Google Colab / Jupyter (Mobile, Windows, or Mac), render the Dark UI directly!
    if IN_NOTEBOOK:
        display(HTML(dashboard_html))

    # Also save a standalone HTML dashboard file that opens in any mobile/desktop browser
    with open("Security_Audit_Dashboard.html", "w", encoding="utf-8") as f:
        f.write(dashboard_html)
    print("\n✅ Audit Complete! Standalone dark-theme report saved as 'Security_Audit_Dashboard.html'.")

🔍 Starting BioShield Web Security Scanner...
 -> Auditing: https://www.ncbi.nlm.nih.gov ...
 -> Auditing: https://github.com ...
 -> Auditing: http://example.com ...


Security Header,Audit Status,Configured Value,Role & Bio-Defense Analogy
Strict-Transport-Security,PRESENT (SAFE),max-age=31536000; includeSubDomains; preload,Enforces encrypted HTTPS connections (HSTS). (Outer Cell Wall: Prevents eavesdropping in transit.)
Content-Security-Policy,PRESENT (SAFE),upgrade-insecure-requests,Blocks unauthorized scripts & Cross-Site Scripting (XSS). (Immune Receptor: Rejects foreign viral DNA/scripts.)
X-Frame-Options,PRESENT (SAFE),SAMEORIGIN,Prevents Clickjacking (hiding the site inside invisible frames). (Structural Cytoskeleton: Stops cell-mimicry traps.)
X-Content-Type-Options,MISSING (Medium Risk),Not Configured,Stops browsers from guessing/sniffing dangerous file types. (Enzyme Specificity: Ensures only exact file types execute.)
Referrer-Policy,PRESENT (SAFE),origin-when-cross-origin,Controls how much user navigation history is leaked to other sites. (Membrane Transport Gate: Limits internal data leakage.)
Permissions-Policy,MISSING (Low Risk),Not Configured,"Restricts unauthorized access to camera, microphone, and GPS. (Organelle Lock: Protects sensitive hardware functions.)"
Security Header,Audit Status,Configured Value,Role & Bio-Defense Analogy
Strict-Transport-Security,PRESENT (SAFE),max-age=31536000; includeSubdomains; preload,Enforces encrypted HTTPS connections (HSTS). (Outer Cell Wall: Prevents eavesdropping in transit.)
Content-Security-Policy,PRESENT (SAFE),default-src 'none'; base-uri 'self'; child-src github.g...,Blocks unauthorized scripts & Cross-Site Scripting (XSS). (Immune Receptor: Rejects foreign viral DNA/scripts.)
X-Frame-Options,PRESENT (SAFE),deny,Prevents Clickjacking (hiding the site inside invisible frames). (Structural Cytoskeleton: Stops cell-mimicry traps.)



✅ Audit Complete! Standalone dark-theme report saved as 'Security_Audit_Dashboard.html'.
